In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 13: Menjalankan model lokal dengan Ollama

Cerita lengkapnya ada di `README.md`.

Mode cepat membuat ulang adapter kecil, menggabungkannya, lalu membandingkan dengan kontrol tanpa LoRA. Konversi GGUF dan impor Ollama tidak dijalankan di gerbang.

Bukan data asli perusahaan. Ollama, bila dinyalakan, hanya di `127.0.0.1`.


In [2]:
# Satu ide: numpy dipakai di mode cepat, tanpa torch dan tanpa unduhan.
import numpy as np


In [3]:
# Satu ide: buat ulang data sintetis yang sama, bukan mengambil berkas adapter.
rng = np.random.default_rng(SEED)
X = rng.normal(size=(40, 4))
arah = np.array([0.7, -0.4, 0.5, 0.2])
y = (X @ arah > 0).astype(float)


In [4]:
# Satu ide: pecah set latih dan set uji sebelum adapter dibuat ulang.
acak = rng.permutation(len(X))
X_latih, y_latih = X[acak[:30]], y[acak[:30]]
X_uji, y_uji = X[acak[30:]], y[acak[30:]]
print("baris_latih", int(len(X_latih)))
print("baris_uji", int(len(X_uji)))


baris_latih 30
baris_uji 10


In [5]:
# Satu ide: bobot dasar dibekukan, catatan kecil mulai dari angka kecil.
W = rng.normal(scale=0.05, size=4)
W_awal = W.copy()
a = rng.normal(scale=0.1, size=4)
b = 0.0


In [6]:
# Satu ide: ukur kontrol sebelum catatan dilatih, pada set uji.
def kerugian(bobot, vektor_a, skalar_b, data_x, data_y):
    delta = bobot + skalar_b * vektor_a
    pred = 1 / (1 + np.exp(-(data_x @ delta)))
    return float(np.mean((pred - data_y) ** 2))
sebelum = kerugian(W, a, b, X_uji, y_uji)
print("kerugian_uji_sebelum", round(sebelum, 6))


kerugian_uji_sebelum 0.246913


In [7]:
# Satu ide: buat ulang adapter LoRA kecil, delapan puluh langkah, di CPU.
for _ in range(80):
    pred = 1 / (1 + np.exp(-(X_latih @ (W + b * a))))
    d = (2 * (pred - y_latih) / len(y_latih)) * pred * (1 - pred)
    g = X_latih.T @ d
    a_baru = a - 0.8 * (b * g)
    b = b - 0.8 * float(a @ g)
    a = a_baru


In [8]:
# Satu ide: set uji yang sama lebih baik sesudah adapter dibuat ulang.
sesudah = kerugian(W, a, b, X_uji, y_uji)
selisih = sebelum - sesudah
print("kerugian_uji_sesudah", round(sesudah, 6))
print("selisih_uji", round(selisih, 6))
print("bobot_tetap", "ya" if np.allclose(W, W_awal) else "tidak")
assert selisih >= 0.15
assert np.allclose(W, W_awal)


kerugian_uji_sesudah 0.050404
selisih_uji 0.196509
bobot_tetap ya


In [9]:
# Satu ide: gabungkan catatan ke bobot, dan siapkan kontrol tanpa LoRA.
W_gabung = W + b * a
W_kontrol = W.copy()
tensor_gabung = int(W_gabung.ndim > 0)
tensor_kontrol = int(W_kontrol.ndim > 0)
print("tensor_gabung", tensor_gabung)
print("tensor_kontrol", tensor_kontrol)
print("jumlah_sama", "ya" if W_gabung.size == W_kontrol.size else "tidak")


tensor_gabung 1
tensor_kontrol 1
jumlah_sama ya


In [10]:
# Satu ide: satu baris yang sama, skor gabungan beda dari kontrol.
skor_gabung = float(X_uji[0] @ W_gabung)
skor_kontrol = float(X_uji[0] @ W_kontrol)
beda = abs(skor_gabung - skor_kontrol)
print("skor_gabung", round(skor_gabung, 6))
print("skor_kontrol", round(skor_kontrol, 6))
print("beda_skor", round(beda, 6))
assert beda >= 0.5


skor_gabung -4.496645
skor_kontrol 0.007784
beda_skor 4.504429


In [11]:
# Satu ide: nama jalur GGUF dan alamat peladen, tanpa menjalankannya.
print("skrip", "convert_hf_to_gguf.py")
print("commit", "ca89968")
print("alamat", "127.0.0.1")
print("impor_ollama", "bukan_gerbang")


skrip convert_hf_to_gguf.py
commit ca89968
alamat 127.0.0.1
impor_ollama bukan_gerbang


In [12]:
# Satu ide: revisi model dipin, dan mode cepat tidak mengunduh.
import importlib.util
import sys
from pathlib import Path
akar = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "phase2-ai" / "muat_hf.py").is_file())
sys.path.insert(0, str(akar / "phase2-ai"))
from muat_hf import REVISI
spec = importlib.util.spec_from_file_location("muat13", akar / "phase2-ai" / "modul-13-model-lokal" / "muat.py")
muat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(muat)
print("model", muat.ID_DASAR)
print("revisi", REVISI[muat.ID_DASAR])
print("model_penuh", muat.ID_PENUH)
print("revisi_penuh", REVISI[muat.ID_PENUH])
print("unduh", False)


model HuggingFaceTB/SmolLM2-135M-Instruct
revisi 12fd25f77366fa6b3b4b768ec3050bf629380bac
model_penuh Qwen/Qwen2.5-0.5B-Instruct
revisi_penuh 7ae557604adf67be50417f59c2c2f167def9a775
unduh False
